In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/paysim.csv")
print(df.shape)
df.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [15]:
print(df.dtypes)
print("\nNulls:\n", df.isnull().sum())
print("\nstep range:", df.step.min(), "to", df.step.max())

step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Nulls:
 step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

step range: 1 to 743


In [16]:
print("Fraud count:", df.isFraud.sum())
print("Fraud rate: %.4f%%" % (df.isFraud.mean() * 100))
print()
print(df.groupby("type").isFraud.agg(["count", "sum", "mean"]))


Fraud count: 8213
Fraud rate: 0.1291%

            count   sum      mean
type                             
CASH_IN   1399284     0  0.000000
CASH_OUT  2237500  4116  0.001840
DEBIT       41432     0  0.000000
PAYMENT   2151495     0  0.000000
TRANSFER   532909  4097  0.007688


In [17]:
print(df.groupby("isFraud").amount.describe())

             count          mean           std   min         25%        50%  \
isFraud                                                                       
0        6354407.0  1.781970e+05  5.962370e+05  0.01   13368.395   74684.72   
1           8213.0  1.467967e+06  2.404253e+06  0.00  127091.330  441423.44   

                75%          max  
isFraud                           
0         208364.76  92445516.64  
1        1517771.48  10000000.00  


In [18]:
print("Unique senders:", df.nameOrig.nunique(), "of", len(df))
print("Unique receivers:", df.nameDest.nunique())
print("\nTop receivers by txn count:")
print(df.nameDest.value_counts().head())
print("\nSenders with more than 1 txn:", (df.nameOrig.value_counts() > 1).sum())

Unique senders: 6353307 of 6362620
Unique receivers: 2722362

Top receivers by txn count:
nameDest
C1286084959    113
C985934102     109
C665576141     105
C2083562754    102
C248609774     101
Name: count, dtype: int64

Senders with more than 1 txn: 9298


In [19]:
fr = df[df.type.isin(["TRANSFER", "CASH_OUT"])]
drained = (fr.newbalanceOrig == 0) & (fr.amount == fr.oldbalanceOrg)
print("Fraud rate when sender fully drained:", fr[drained].isFraud.mean())
print("Fraud rate otherwise:", fr[~drained].isFraud.mean())
print("\nisFlaggedFraud vs isFraud:")
print(pd.crosstab(df.isFlaggedFraud, df.isFraud))

Fraud rate when sender fully drained: 1.0
Fraud rate otherwise: 6.841913781026179e-05

isFlaggedFraud vs isFraud:
isFraud               0     1
isFlaggedFraud               
0               6354407  8197
1                     0    16
